## Part 09. 프로젝트 루트 확인과 4개 CSV 불러오기


35. 프로젝트 루트 설정
[출처] Chapter 04. pandas로 데이터에 질문하기|작성자 아토믹데브

In [337]:

from pathlib import Path
project_root = Path.cwd()

if project_root.name == "notebooks":

    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)

print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [338]:

import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")


## 37. 기본 구조와 주요 키 확인

In [339]:

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [340]:
key_checks = {

    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():

    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## Part 10. 컬럼 선택·조건 필터링·정렬

## 38. Series와 DataFrame 선택

In [341]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 39. 단일 조건 필터링

In [342]:

customers_over_30 = customers[
    customers["age"] >= 30
]

print(len(customers), len(customers_over_30))
display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-30
2,3,이경수,F,61,성남,2024-07-10
3,4,조영호,F,55,울산,2026-05-11
5,6,김지원,F,32,성남,2026-07-25
6,7,이상현,F,53,인천,2025-01-09


## 40. 복합 조건 필터링

# 30세 이상이면서 서울 거주:

In [343]:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]

In [344]:

seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)

city
부산    16
서울    15
Name: count, dtype: int64

### 완료주문이 아닌 주문 (~ 이게 무엇무엇이 아닌이란 듯)

In [345]:
not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(

    not_completed["order_status"].value_counts(
        dropna=False
    )

)

order_status
cancelled    65
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [346]:
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)

display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## Part 11. line_total 생성과 전체 주문 금액 구분

## 42. 작업용 복사본과 파생 컬럼

In [347]:
order_items_work = order_items.copy()

# 데이터 플레임에 새로운 컬럼을 추가할 떄는 기존 데이터 프레임을 수정
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]

)

In [348]:
display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


## 43. 수작업 검증

In [349]:
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True



## 44. 전체 주문상세 금액

In [350]:
all_order_amount = order_items_work["line_total"].sum()
print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255610000



현재 합계에는 취소 또는 환불 주문이 포함될 수 있으므로
완료 주문 기준 매출이 아니라 전체 주문상세 금액으로 표현한다.

## Part 12. 주문 데이터 병합과 완료 주문 분석셋 만들기

#### 45. 병합용 주문 컬럼 선택

In [351]:
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

(301, 4)
   order_id  customer_id  order_date order_status
0         1          123  2026-06-04    completed
1         2           77  2025-08-20    cancelled
2         3          138  2025-12-17    cancelled
3         4           57  2026-02-27    cancelled
4         5          125  2026-01-18    cancelled


## 46. 주문상세와 주문 병합

In [352]:
# order_items와 orders_for_merge의 차이점은 파생 column line_total이 존재하는지 여부이다. 
# order_items_work에는 line_total이 존재하지만 orders_for_merge에는 존재하지 않는다.

order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

## 47. 병합 검증

In [353]:

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 764
병합 후 행 수: 764


order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [354]:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


## 48. 완료 주문 분석셋

In [355]:
# value_counts()는 각 값이 몇 번 등장하는지 센다

display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [356]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [357]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)

print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)

print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## Part 13. 상품 데이터 병합과 카테고리·상품 매출

## 49. 필요한 상품 정보만 선택

In [358]:

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

print(products_for_merge.head())

   product_id product_name category
0           1  전자기기 상품 001     전자기기
1           2    도서 상품 002       도서
2           3  전자기기 상품 003     전자기기
3           4  생활용품 상품 004     생활용품
4           5    식품 상품 005       식품


## 50. 완료 주문상세와 상품 병합

In [359]:

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [360]:

print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

### 51. 카테고리별 매출

In [361]:

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


## 52. 카테고리 합계 검증

In [362]:
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


## 53. 상품별 매출

In [363]:
# sort values = 정렬
# groupy = 그룹화


product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


## Part 14. 월별 매출과 고객별 구매 금액

### 54. 주문 날짜 변환과 주문 월 생성

In [364]:
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [365]:

completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

## 55. 월별 매출

In [366]:

monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,5869000,8,8,52
1,2025-09,16147000,19,19,132
2,2025-10,12385000,15,15,120
3,2025-11,23550000,24,23,233
4,2025-12,9876000,13,13,99
5,2026-01,10851000,13,13,105
6,2026-02,16504000,21,20,150
7,2026-03,9885000,18,16,102
8,2026-04,15536000,17,16,157
9,2026-05,15310000,19,18,152


## 56. 고객별 구매 금액

In [367]:

customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

## 57. 고객 속성 연결

#### 개인정보 최소화를 위해 이름은 제외합니다.

In [368]:
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [369]:

customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [370]:
print(customer_sales.sort_values("total_sales", ascending=False).head(10)
)
#.sort_values("total_sales", ascending=False))
#display(product_sales.head(10))

    customer_id  total_sales  order_count  quantity_sold
76          117      4100000            5             48
62          102      3996000            4             35
51           83      3880000            4             39
21           30      3590000            5             32
29           40      3523000            4             27
13           20      3191000            2             25
0             3      3178000            2             26
70          111      3153000            3             38
42           66      3093000            4             30
97          147      2990000            2             21


In [371]:

display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
76,117,4100000,5,48,F,65,성남,both
62,102,3996000,4,35,M,60,고양,both
51,83,3880000,4,39,F,22,수원,both
21,30,3590000,5,32,F,32,서울,both
29,40,3523000,4,27,M,23,서울,both
13,20,3191000,2,25,F,20,인천,both
0,3,3178000,2,26,F,61,성남,both
70,111,3153000,3,38,F,41,광주,both
42,66,3093000,4,30,F,39,서울,both
97,147,2990000,2,21,M,19,부산,both


## Part 15. 결과 CSV 저장과 반복 점검 함수

##### 58. 결과 폴더 생성

In [372]:
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

c:\dev\ai-data-analysis\reports\chapter04


## 59. 결과 파일 저장

In [373]:

outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 309
product_sales.csv True 4833
monthly_sales.csv True 415
customer_sales.csv True 3448


## 60. 저장 결과 다시 읽기

In [374]:

saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


## 61. 병합 점검 함수

In [375]:

def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

In [376]:
#위에 만든 함수 호출

check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 764
병합 후 행 수: 764
order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64


## 62. 집계 합계 검증 함수

In [377]:

def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
    
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [378]:
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 148990000
요약 합계: 148990000
차이: 0


64. pandas 코드 요청 프롬프트

 

나는 온라인 쇼핑몰 데이터를 pandas로 분석하고 있습니다.

분석 목표:

완료 주문 기준 카테고리별 매출을 계산합니다.

DataFrame과 한 행의 의미:

- orders: 주문 한 건
- order_items: 주문에 포함된 상품 한 항목
- products: 상품 한 개

실제 컬럼:

- orders:
  order_id, customer_id, order_date,
  payment_method, order_status

- order_items:
  order_item_id, order_id, product_id,
  quantity, unit_price

- products:
  product_id, product_name, category, price

주요 관계:
- order_items.order_id → orders.order_id
  many_to_one
- order_items.product_id → products.product_id
  many_to_one

분석 범위:
- order_status가 completed인 주문만 포함
- line_total = quantity × unit_price
- 주문 수는 order_id의 고유 개수
- 매출은 line_total 합계

원하는 결과:
- category
- total_sales
- order_count
- customer_count
- quantity_sold

검증 요구사항:
1. 각 merge에 validate를 사용해 주세요.
2. indicator로 미매칭을 확인해 주세요.
3. 병합 전후 행 수를 출력해 주세요.
4. 카테고리 합계와 완료 주문 전체 합계를 비교해 주세요.
5. 실제로 존재하지 않는 컬럼을 만들지 마세요.
6. 코드 실행 전 확인할 항목도 설명해 주세요.

In [388]:
# LLM 출력 코드

# ============================================
# 0. 코드 실행 전 데이터 확인
# ============================================

print("orders 컬럼:")
print(orders.columns.tolist())

print("\norder_items 컬럼:")
print(order_items.columns.tolist())

print("\nproducts 컬럼:")
print(products.columns.tolist())

print("\n주문 상태값:")
print(
    orders["order_status"]
    .value_counts(dropna=False)
)

print("\norders.order_id 중복 개수:")
print(orders["order_id"].duplicated().sum())

print("\nproducts.product_id 중복 개수:")
print(products["product_id"].duplicated().sum())


# ============================================
# 1. 주문 상세 데이터 복사
# ============================================

order_items_work = order_items.copy()


# ============================================
# 2. line_total 계산
# quantity × unit_price
# ============================================

order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)


# ============================================
# 3. orders에서 필요한 컬럼만 선택
# ============================================

orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()


# ============================================
# 4. order_items + orders 병합
# ============================================

order_sales = order_items_work.merge(
    orders_for_merge,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="order_match",
)


# ============================================
# 5. 주문 병합 검증
# ============================================

print("\n=== 주문 병합 검증 ===")

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))

print("\n매칭 결과:")
print(
    order_sales["order_match"]
    .value_counts(dropna=False)
)


# ============================================
# 6. completed 주문만 선택
# ============================================

completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()


print("\n완료 주문상세 행 수:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique()
)


# ============================================
# 7. products에서 필요한 컬럼만 선택
# ============================================

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()


# ============================================
# 8. 완료 주문 + 상품 정보 병합
# ============================================

completed_items = completed_sales.merge(
    products_for_merge,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_match",
)


# ============================================
# 9. 상품 병합 검증
# ============================================

print("\n=== 상품 병합 검증 ===")

print("병합 전 행 수:", len(completed_sales))
print("병합 후 행 수:", len(completed_items))

print("\n매칭 결과:")
print(
    completed_items["product_match"]
    .value_counts(dropna=False)
)


# ============================================
# 10. 카테고리별 매출 계산
# ============================================

category_sales = (
    completed_items
    .groupby(
        "category",
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values(
        "total_sales",
        ascending=False,
    )
)


# ============================================
# 11. 최종 결과 확인
# ============================================

display(category_sales)


# ============================================
# 12. 최종 합계 검증
# ============================================

completed_total = completed_items["line_total"].sum()

category_total = category_sales["total_sales"].sum()


print("\n=== 최종 합계 검증 ===")

print("완료 주문 전체 매출:", completed_total)

print("카테고리별 매출 합계:", category_total)

print(
    "합계 일치 여부:",
    completed_total == category_total
)

orders 컬럼:
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

order_items 컬럼:
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']

products 컬럼:
['product_id', 'product_name', 'category', 'price']

주문 상태값:
order_status
completed    184
cancelled     65
refunded      52
Name: count, dtype: int64

orders.order_id 중복 개수:
0

products.product_id 중복 개수:
0

=== 주문 병합 검증 ===
병합 전 행 수: 764
병합 후 행 수: 764

매칭 결과:
order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64

완료 주문상세 행 수: 474
완료 주문 수: 184

=== 상품 병합 검증 ===
병합 전 행 수: 474
병합 후 행 수: 474

매칭 결과:
product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64


,category,total_sales,order_count,customer_count,quantity_sold
3,스포츠,31743000,85,67,295
5,전자기기,26400000,60,44,259
2,생활용품,23915000,65,50,272
1,뷰티,23383000,65,53,223
4,식품,16573000,36,31,133
0,도서,16389000,52,46,149
6,패션,10587000,33,27,111



=== 최종 합계 검증 ===
완료 주문 전체 매출: 148990000
카테고리별 매출 합계: 148990000
합계 일치 여부: True


## 65. LLM 코드 검증

### 1. DataFrame 검증

LLM 코드에서 사용한 DataFrame은 `orders`, `order_items`, `products`이다.
실제 Notebook에서 해당 변수들이 존재하는지 확인하였다.

In [389]:
# 사용한 코드는 다음과 같다

print(type(orders))
print(type(order_items))
print(type(products))

<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>


확인 결과 세 변수 모두 pandas DataFrame으로 생성되어 있었다.

결과: 이상 없음

## 2. 컬럼 검증

LLM이 작성한 코드에서 사용한 컬럼이 실제 DataFrame에 존재하는지 확인하였다.
확인에 사용한 코드는 다음과 같다.

In [382]:
print("orders:", orders.columns.tolist())
print("order_items:", order_items.columns.tolist())
print("products:", products.columns.tolist())

orders: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
products: ['product_id', 'product_name', 'category', 'price']


실제 컬럼 목록과 비교한 결과, LLM 코드에서 사용한 컬럼이 모두 존재하였다.

결과: 이상 없음

## 3. 상태값 검증

LLM 코드에서는 완료 주문을 필터링하기 위해
`order_status == "completed"` 조건을 사용하였다.

In [387]:
# 실제 `orders` 데이터의 상태값을 확인하기 위해 다음 코드를 실행하였다

print(
    orders["order_status"]
    .value_counts(dropna=False)
)

order_status
completed    184
cancelled     65
refunded      52
Name: count, dtype: int64


실행 결과 completed 상태값이 실제 데이터에 존재하는 것을 확인하였다.

따라서 LLM 코드에서 사용한 완료 주문 필터 조건은 실제 데이터와 일치한다.

결과: 이상 없음

### 4. 계산식 검증

LLM 코드에서는 주문상세 금액을
`line_total = quantity × unit_price`로 계산하였다.
계산식이 올바르게 적용되었는지 첫 번째 행을 기준으로 직접 계산한 값과
`line_total` 값을 비교하였다.

In [390]:
sample = order_items_work.iloc[0]

expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]

print("직접 계산:", expected)
print("line_total:", actual)
print("일치 여부:", expected == actual)

직접 계산: 306000
line_total: 306000
일치 여부: True


실행 결과 직접 계산한 값과 line_total 값이 일치하였다.

결과: 이상없음

### 5. 분석 범위 검증

LLM 코드에서는 완료 주문만 분석에 포함하기 위해
`order_status == "completed"` 조건으로 필터링하였다.

필터링 이후 실제로 완료 주문만 남아 있는지 확인하였다.

In [391]:
print(
    completed_sales["order_status"]
    .value_counts(dropna=False)
)

order_status
completed    474
Name: count, dtype: int64


실행 결과 completed 상태만 존재하는 것을 확인하였다.

따라서 분석 범위가 완료 주문으로 올바르게 제한되었다.

결과: 이상없음

### 6. 주문 수 계산 방식 검증

LLM 코드에서는 주문 수를 계산할 때
`order_id`의 고유 개수를 구하는 `nunique()`를 사용하였다.

In [392]:
print("주문상세 행 수:", len(completed_sales))
print("실제 주문 수:", completed_sales["order_id"].nunique())

주문상세 행 수: 474
실제 주문 수: 184


주문상세 행 수는 474개였지만, `order_id`의 고유 개수는 184개였다.
따라서 주문 수 계산에는 단순 행 수가 아니라 `nunique()`를 사용하는 것이 적절하다.

- 결과: 이상 없음

### 7. 병합 키와 관계 수 검증

LLM 코드에서는 다음 키를 기준으로 병합하였다.
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`

두 관계 모두 `many_to_one`으로 설정하였다.
오른쪽 기준 테이블의 키가 고유한지 확인하였다.

In [393]:
print(
    "orders.order_id 중복:",
    orders["order_id"].duplicated().sum()
)
print(
    "products.product_id 중복:",
    products["product_id"].duplicated().sum()
)

orders.order_id 중복: 0
products.product_id 중복: 0


orders.order_id와 products.product_id의 중복 개수가 모두 0이라면,
오른쪽 기준 키가 고유하므로 many_to_one 관계가 적절하다.

결과: 이상없음

### 8. validate 검증

LLM 코드의 두 번의 병합에서 모두
`validate="many_to_one"`이 사용되었는지 확인하였다.

In [394]:
# 첫번째 병합

order_sales = order_items_work.merge(
    orders_for_merge,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="order_match",
)

In [395]:
# 두번째 병합

completed_items = completed_sales.merge(
    products_for_merge,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_match",
)

두 병합 모두 many_to_one 관계를 검증하도록 설정되어 있었다.

결과: 이상없음

### 9. indicator 검증

LLM 코드에서는 병합 결과의 미매칭 여부를 확인하기 위해
각 merge에 `indicator`를 사용하였다.

In [396]:
print(
    order_sales["order_match"]
    .value_counts(dropna=False)
)

print(
    completed_items["product_match"]
    .value_counts(dropna=False)
)

order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64
product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64


order_match와 product_match의 값을 확인하여
병합된 데이터가 양쪽 테이블에 모두 존재하는지 확인하였다.

모든 행이 both라면 정상적으로 병합된 것으로 판단할 수 있다.

결과: 이상 없음

### 10. 병합 전후 행 수 검증

LLM 코드에서 수행한 두 번의 병합 전후 행 수를 비교하였다.

In [397]:
print("1차 병합 전:", len(order_items_work))
print("1차 병합 후:", len(order_sales))

print("2차 병합 전:", len(completed_sales))
print("2차 병합 후:", len(completed_items))

1차 병합 전: 764
1차 병합 후: 764
2차 병합 전: 474
2차 병합 후: 474


many_to_one 관계의 병합에서는 일반적으로 병합 전후 행 수가 유지되어야 한다.

실행 결과 두 병합 모두 병합 전후 행 수가 동일하여,
예상하지 못한 행 증가가 발생하지 않은 것을 확인하였다.

결과: 이상 없음

### 11. 합계 검증

LLM 코드에서 계산한 카테고리별 매출 합계가
완료 주문 전체 매출과 일치하는지 확인하였다.

In [398]:
completed_total = completed_items["line_total"].sum()
category_total = category_sales["total_sales"].sum()

print("완료 주문 전체 매출:", completed_total)
print("카테고리별 매출 합계:", category_total)
print("합계 일치 여부:", completed_total == category_total)

완료 주문 전체 매출: 148990000
카테고리별 매출 합계: 148990000
합계 일치 여부: True


실행 결과 완료 주문 전체 매출과 카테고리별 매출 합계가 일치하였다.

따라서 카테고리별 집계 과정에서 매출 누락이나 중복이 발생하지 않은 것으로 확인하였다.

결과: 이상 없음

### 12. 개인정보 검증

LLM이 작성한 코드에서 분석에 불필요한 고객 개인정보를
사용하거나 요구하는지 확인하였다.

카테고리별 매출 분석에는 고객을 구분하기 위한 `customer_id`만 사용하였으며,
고객 이름, 이메일, 전화번호, 주소 등의 개인정보는 사용하지 않았다.

따라서 분석에 필요한 최소한의 고객 정보만 사용하였다.

- 결과: 이상 없음

## 65. LLM 코드 검증표

| 검증 항목 | 확인 내용 | 결과 |
|---|---|---|
| DataFrame | 실제 변수명과 같은가? | `orders`, `order_items`, `products` 모두 실제 Notebook의 DataFrame과 일치함 |
| 컬럼 | 실제 컬럼만 사용하는가? | LLM 코드에서 사용한 컬럼이 실제 DataFrame에 모두 존재함 |
| 상태값 | `completed` 표기가 맞는가? | `order_status`의 실제 값에 `completed`가 존재함 |
| 계산식 | `quantity × unit_price`인가? | `line_total = quantity × unit_price`로 계산되며 직접 계산값과 일치함 |
| 분석 범위 | 완료 주문만 포함하는가? | 필터링 후 `completed` 상태만 남아 있음을 확인함 |
| 주문 수 | `nunique()`를 사용하는가? | 주문상세 474행에 대해 고유 `order_id`는 184개로 확인되어 `nunique()` 사용이 적절함 |
| 병합 키 | 실제 관계와 맞는가? | `order_id`, `product_id`를 기준으로 실제 데이터 관계에 맞게 병합함 |
| `validate` | `many_to_one`이 적용되었는가? | 두 번의 `merge()` 모두 `validate="many_to_one"`을 사용함 |
| `indicator` | 미매칭을 확인하는가? | `order_match`, `product_match`를 확인한 결과 모두 `both`로 나타나 미매칭이 없음 |
| 행 수 | 병합 전후를 비교하는가? | 두 번의 병합 모두 병합 전후 행 수가 동일하여 예상하지 못한 행 증가가 없음 |
| 합계 | 원본과 요약 합계를 비교하는가? | 완료 주문 전체 매출과 카테고리별 매출 합계가 동일하며 비교 결과 `True`로 확인됨 |
| 개인정보 | 원본 고객 정보를 요구하지 않는가? | `customer_id`만 사용하고 이름, 이메일, 전화번호, 주소 등의 개인정보는 사용하지 않음 |